# Exhaustive saved v3 verification and independent-review handoff

Every saved parent/window, label, magnitude mask, identity and eligibility is checked. A separate equation implementation reproduces stratified earthquake parents; source replay covers all noise families. This is implementation validation, not final independent approval. No CNN training. All writes stay inside v3.


In [1]:
from pathlib import Path
import json,sys,hashlib,ast
sys.dont_write_bytecode=True
V3=Path(r"C:/Users/choud/Desktop/Minor project/cnn/cnn wave/eew_causal_v3")
sim_nb=json.loads((V3/"01_causal_simulator_v3.ipynb").read_text(encoding="utf-8"))
exec(compile("".join(sim_nb["cells"][1]["source"]),"v3_definitions_for_verification","exec"))
generation=readj(V3/"configs/generation_manifest.json")
assert generation["status"]=="COMPLETE"
assert generation["simulator_notebook_sha256"]==sha(V3/"01_causal_simulator_v3.ipynb")
assert generation["definition_hash"]==hashlib.sha256("".join(sim_nb["cells"][1]["source"]).encode()).hexdigest()
assert generation["configuration_sha256"]==sha(V3/"configs/simulation_v3.json")
assert all(sha(V3/"dataset"/f"eew_dataset_{split}.h5")==digest for split,digest in generation["files"].items())
SIM=ROOT/"data wave/synthetic_waveform_simulator/output"
metadata=pd.read_csv(SIM/"phase2_travel_time/metadata_with_theoretical_arrivals.csv").set_index("source_id")
assignments=pd.read_csv(SIM/"phase11_preprocessed_dataset/split_indices.csv")
noise_mapping=pd.read_csv(SIM/"phase10_classification_dataset/earthquake_noise_mapping.csv").set_index("combined_index")
manifest=pd.read_csv(V3/"configs/split_manifest.csv",keep_default_na=False)
validation={"splits":{},"replays":[],"rounding_cases":[],"failures":[],"duplicate_parent_cross_split":[],
            "duplicate_background_cross_split":[],"duplicate_window_cross_split":[]}
parent_owner={};background_owner={};window_owner={};frames=[];examples=[]
print("Saved dataset hashes verified; exhaustive validation starting.")


Root causes mapped; fixed gains {'band': 0.7043400785612605, 'pink': 1.3103644084803956} ; original protected files 44
Saved dataset hashes verified; exhaustive validation starting.


In [2]:
for split,n in [("train",11324),("validation",2426),("test",2428)]:
    with h5py.File(V3/"dataset"/f"eew_dataset_{split}.h5","r") as f:
        keys=["labels","magnitude","magnitude_mask","depth_km","hypocentral_distance_km","source_latitude","source_longitude",
              "station_latitude","station_longitude","p_arrival_sample","s_arrival_sample","reference_sample",
              "p_arrival_seconds","s_arrival_seconds","sp_seconds","combined_index","pair_id","seed","context_mean","context_std"]
        for t in [1,2,3]:keys.extend([f"pre_s_valid_{t}s",f"classification_mask_{t}s",f"magnitude_mask_{t}s"])
        d={k:f[k][:] for k in keys}
        for k in ["source_id","trace_name","receiver_code","origin_time_utc","parent_id","background_id","noise_profile","parent_sha256","background_sha256"]:
            d[k]=f[k].asstr()[:]
        df=pd.DataFrame(d);df["split"]=split;df["row"]=np.arange(n);eq=df.labels.to_numpy()==1
        original=assignments[assignments.split==split].reset_index(drop=True)
        m=metadata.loc[df.loc[eq,"source_id"]]
        man=manifest[manifest.split==split].reset_index(drop=True)
        checks={"count":len(df)==n,"balanced":int(eq.sum())==n//2 and set(df.labels)=={0,1},
            "sampling":float(f.attrs["sampling_rate_hz"])==100.,
            "parent_shape_dtype":f["parent_raw"].shape==(n,10000) and f["parent_raw"].dtype==np.dtype("float32"),
            "magnitude_mask":np.array_equal(df.magnitude_mask,eq),
            "magnitude_mask_dtype":f["magnitude_mask"].dtype==np.dtype(bool),
            "noise_targets":bool((df.loc[~eq,"magnitude"]==-1).all()),
            "noise_no_arrival_labels":bool((df.loc[~eq,["p_arrival_sample","s_arrival_sample","p_arrival_seconds","s_arrival_seconds","sp_seconds"]]==-1).all().all()),
            "source_target_magnitudes":np.array_equal(df.loc[eq,"magnitude"].to_numpy(),m.Magnitude_Value.to_numpy(np.float32)),
            "source_depths":np.array_equal(df.loc[eq,"depth_km"],m.Depth_km),
            "source_distance":np.array_equal(df.loc[eq,"hypocentral_distance_km"],m.hypocentral_distance_km),
            "source_coords":np.array_equal(df.loc[eq,"source_latitude"],m.Lat) and np.array_equal(df.loc[eq,"source_longitude"],m.Long),
            "station_coords":np.array_equal(df.loc[eq,"station_latitude"],m.receiver_latitude) and np.array_equal(df.loc[eq,"station_longitude"],m.receiver_longitude),
            "assignment_order":np.array_equal(df.combined_index,original.combined_index) and np.array_equal(df.pair_id,original.pair_id) and np.array_equal(df.labels,original.label),
            "manifest_row_identity":df.parent_id.tolist()==man.parent_id.tolist() and df.source_id.tolist()==man.source_id.tolist(),
            "source_ids":df.loc[eq,"source_id"].tolist()==original.loc[eq,"source_id"].tolist(),
            "source_timestamps":df.loc[eq,"origin_time_utc"].tolist()==m.origin_time_utc.astype(str).tolist(),
            "P_reference_alignment":np.array_equal(df.loc[eq,"reference_sample"],df.loc[eq,"p_arrival_sample"]),
            "P_seconds":np.array_equal(df.loc[eq,"p_arrival_seconds"],df.loc[eq,"p_arrival_sample"]/100),
            "continuous_SP_preserved":np.array_equal(df.loc[eq,"sp_seconds"],m.theoretical_sp_time_seconds),
            "continuous_S_preserved":np.array_equal(df.loc[eq,"s_arrival_seconds"],df.loc[eq,"p_arrival_seconds"]+m.theoretical_sp_time_seconds.to_numpy()),
            "S_first_grid_sample":np.array_equal(df.loc[eq,"s_arrival_sample"],np.ceil(df.loc[eq,"s_arrival_seconds"]*100).astype(np.int32)),
            "P_before_S":bool((df.loc[eq,"p_arrival_sample"]<df.loc[eq,"s_arrival_sample"]).all())}
        checks["earthquake_seeds"]=np.array_equal(df.loc[eq,"seed"],42+df.loc[eq,"combined_index"])
        checks["noise_seeds"]=np.array_equal(df.loc[~eq,"seed"],noise_mapping.loc[df.loc[~eq,"combined_index"],"seed"])
        checks["noise_profiles"]=df.loc[~eq,"noise_profile"].tolist()==noise_mapping.loc[df.loc[~eq,"combined_index"],"noise_profile_type"].tolist()
        refs_expected=np.array([int(round((3+4*(np.random.default_rng(int(seed)).uniform() if label else np.random.default_rng(np.random.SeedSequence([int(seed),22,0])).uniform()))*100)) for seed,label in zip(df.seed,df.labels)])
        checks["seeded_reference_indices"]=np.array_equal(df.reference_sample,refs_expected)
        eligibility={}
        for t,L in [(1,200),(2,300),(3,400)]:
            expected=eq&(df.reference_sample.to_numpy()+100*t<df.s_arrival_sample.to_numpy())
            for key,exp in [(f"pre_s_valid_{t}s",expected),(f"classification_mask_{t}s",~eq|expected),(f"magnitude_mask_{t}s",eq&expected)]:
                checks[key]=bool(np.array_equal(df[key],exp) and f[key].dtype==np.dtype(bool))
            checks[f"true_masks_strict_continuous_{t}"]=bool(np.all(df.loc[expected,"p_arrival_seconds"]+t<df.loc[expected,"s_arrival_seconds"]))
            eligibility[str(t)]={"earthquake_eligible":int(expected.sum()),"earthquakes":int(eq.sum()),
                                "classification_eligible":int((~eq|expected).sum()),"regression_eligible":int((eq&expected).sum())}
            for kind in ["raw","norm"]:
                checks[f"shape_dtype_{t}_{kind}"]=f[f"windows_{t}s_{kind}"].shape==(n,L) and f[f"windows_{t}s_{kind}"].dtype==np.dtype("float32")
            checks[f"shape_dtype_post{t}"]=f[f"windows_pure_post_p_{t}s"].shape==(n,100*t) and f[f"windows_pure_post_p_{t}s"].dtype==np.dtype("float32")
        finite=True;parent_match=True;norm_match=True;context_match=True;hash_match=True;post_match=True
        max_norm_diff=0.
        for key in f:
            if f[key].dtype.kind in "biuf":
                for start in range(0,n,256):finite &= bool(np.isfinite(f[key][start:start+256]).all())
        for start in range(0,n,128):
            end=min(start+128,n);parents=f["parent_raw"][start:end]
            refs=df.reference_sample.to_numpy()[start:end]
            pre=np.stack([x[r-100:r] for x,r in zip(parents,refs)])
            means=pre.mean(axis=1);stds=pre.std(axis=1)+1e-8
            context_match &= np.array_equal(means.astype(float),df.context_mean.to_numpy()[start:end])
            # Generator computes float(std_float32)+1e-8, not float32 vector addition.
            stds=pre.std(axis=1).astype(float)+1e-8
            context_match &= np.array_equal(stds,df.context_std.to_numpy()[start:end])
            for j,parent in enumerate(parents):
                index=start+j;digest=hashlib.sha256(parent.tobytes()).hexdigest()
                hash_match &= digest==df.parent_sha256.iloc[index]
                if digest in parent_owner and parent_owner[digest][0]!=split:
                    validation["duplicate_parent_cross_split"].append([parent_owner[digest],[split,index]])
                parent_owner[digest]=[split,index]
                bg=df.background_sha256.iloc[index]
                if bg in background_owner and background_owner[bg][0]!=split:
                    validation["duplicate_background_cross_split"].append([background_owner[bg],[split,index]])
                background_owner[bg]=[split,index]
            for t,L in [(1,200),(2,300),(3,400)]:
                raw=f[f"windows_{t}s_raw"][start:end];norm=f[f"windows_{t}s_norm"][start:end]
                expected=np.stack([x[r-100:r+t*100] for x,r in zip(parents,refs)])
                parent_match &= np.array_equal(raw,expected)
                post_match &= np.array_equal(f[f"windows_pure_post_p_{t}s"][start:end],raw[:,100:])
                # Scalar-by-scalar float32 arithmetic reproduces generation, independently from saved normalizer values.
                calculated=np.stack([((r-float(mu))/(float(sd))).astype(np.float32) for r,mu,sd in zip(raw,means,stds)])
                norm_match &= np.array_equal(norm,calculated)
                max_norm_diff=max(max_norm_diff,float(np.max(np.abs(norm-calculated))))
                for local,r in enumerate(raw):
                    key=(t,hashlib.sha256(r.tobytes()).hexdigest())
                    if key in window_owner and window_owner[key][0]!=split:validation["duplicate_window_cross_split"].append([window_owner[key],[split,start+local],t])
                    window_owner[key]=[split,start+local]
        checks.update(all_numeric_finite=finite,all_windows_match_parents=parent_match,all_post_windows_match=post_match,
                      all_normalized_windows_match=norm_match,all_context_stats_match=context_match,all_parent_hashes_match=hash_match)
        for t in [1,2,3]:
            mask=df[f"magnitude_mask_{t}s"].to_numpy();target=df.magnitude.to_numpy();poison=target.copy();poison[~eq]=1e9
            checks[f"regression_noise_poison_exclusion_{t}"]=magnitude_mae(np.zeros(n),target,mask)==magnitude_mae(np.zeros(n),poison,mask)
        positive=df[eq]
        pick=set([positive.magnitude.idxmin(),positive.magnitude.idxmax(),positive.depth_km.idxmin(),positive.depth_km.idxmax(),positive.sp_seconds.idxmin()])
        pick.update(df.index[df.source_id.isin(["NCS_EV_03038","NCS_EV_03572"])].tolist())
        for profile,group in df[~eq].groupby("noise_profile"):pick.add(int(group.index[0]))
        for i in sorted(pick):
            row=df.iloc[i].to_dict()
            row["saved_parent"]=f["parent_raw"][i];row["saved_norm"]=f["windows_3s_norm"][i]
            examples.append(row)
        for row in df[df.source_id.isin(["NCS_EV_03038","NCS_EV_03572"])].itertuples():
            validation["rounding_cases"].append({"source_id":row.source_id,"SP_seconds":row.sp_seconds,
                "v2_nearest_SP_samples":int(np.rint(row.s_arrival_seconds*100))-row.p_arrival_sample,
                "v3_first_grid_SP_samples":row.s_arrival_sample-row.p_arrival_sample,
                "eligible2":bool(row.pre_s_valid_2s),"eligible3":bool(row.pre_s_valid_3s)})
        validation["splits"][split]={"count":n,"earthquakes":int(eq.sum()),"checks":checks,"eligibility":eligibility,
                                     "max_normalization_difference":max_norm_diff,
                                     "P_min_max":[int(positive.p_arrival_sample.min()),int(positive.p_arrival_sample.max())]}
        validation["failures"].extend([f"{split}:{k}" for k,v in checks.items() if not v])
        frames.append(df)
df_all=pd.concat(frames,ignore_index=True)
overlap={}
for a,b in [("train","validation"),("train","test"),("validation","test")]:
    left=df_all[df_all.split==a];right=df_all[df_all.split==b]
    for key in ["source_id","parent_id","background_id","pair_id"]:
        overlap[f"{a}/{b}:{key}"]=len(set(left[key])&set(right[key]))
validation["membership_overlap"]=overlap
pairs=df_all.groupby("pair_id").agg(records=("labels","size"),positive=("labels","sum"),splits=("split","nunique"))
validation["pair_membership_valid"]=bool(((pairs.records==2)&(pairs.positive==1)&(pairs.splits==1)).all())
assert not validation["failures"],"CRITICAL saved-data check failure: "+str(validation["failures"])
assert not any(overlap.values()) and validation["pair_membership_valid"]
assert not validation["duplicate_parent_cross_split"] and not validation["duplicate_background_cross_split"] and not validation["duplicate_window_cross_split"]
print("Exhaustive PASS:",len(df_all),"records; all parent/window relations and masks verified.")


Exhaustive PASS: 16178 records; all parent/window relations and masks verified.


In [3]:
# Separate equation-level reconstruction, independent of the transformed earthquake class.
def independent_parent(magnitude,distance,SP,seed):
    n=10000;t=np.arange(n,dtype=np.float64)*.01
    p=int(np.round((3+4*np.random.default_rng(seed).uniform())*100))
    tp=p/100;ts=tp+SP;R=max(distance,params["ref_distance_km"])
    source=params["ref_amplitude"]*10**(params["mag_scaling_coeff"]*(magnitude-params["ref_magnitude"]))
    geom=(params["ref_distance_km"]/R)**params["geometric_spreading_exp"]
    fp=params["p_dominant_freq_hz"];fs=params["s_dominant_freq_hz"]
    ap=source*geom*np.exp(-np.pi*fp*R/(params["qp_0"]*fp**params["q_eta"]*params["vp_km_s"]))
    ass=source*geom*np.exp(-np.pi*fs*R/(params["qs_0"]*fs**params["q_eta"]*params["vs_km_s"]))*params["sp_amp_ratio_bhz"]
    dp=max(params["tau_p_base_s"]*(1+.12*(magnitude-params["ref_magnitude"]))*(1+.001*R),.6);rp=.28*dp
    ds=max(params["tau_s_base_s"]*(1+.18*(magnitude-params["ref_magnitude"]))*(1+.0015*R),1.5);rs=.22*ds
    pw=np.zeros(n);sw=np.zeros(n)
    ix=t>=tp;tau=t[ix]-tp
    env=(tau/rp)**2*np.exp(-tau/dp);env/=((2*dp/rp)**2*np.exp(-2))
    pw[ix]=ap*env*(.85*np.sin(2*np.pi*fp*tau)+.15*np.sin(4*np.pi*fp*tau))
    ix=t>=ts;tau=t[ix]-ts
    env=(tau/rs)**2*np.exp(-tau/ds);env/=((2*ds/rs)**2*np.exp(-2))
    sw[ix]=ass*env*(.80*np.sin(2*np.pi*fs*tau+np.pi/4)+.20*np.sin(3*np.pi*fs*tau))
    innovations=np.random.default_rng(np.random.SeedSequence([int(seed),11,1])).normal(0.,1.,n)
    calibrated=(sosfilt(sos,innovations)*(.05/GAINS["band"])).astype(np.float32)
    return (pw+sw).astype(np.float32)+calibrated,calibrated,p
# Fixed float32 comparison criterion specified independently of observed reconstruction residuals.
reference_atol=1e-7;reference_rtol=1e-6
for row in examples:
    if row["labels"]:
        m=metadata.loc[row["source_id"]]
        parent,bg,reference=independent_parent(float(m.Magnitude_Value),float(m.hypocentral_distance_km),
                                              float(m.theoretical_sp_time_seconds),int(row["seed"]))
        kind="separate earthquake equation implementation"
    else:
        out=CausalNoiseWindowGeneratorV3(params).generate_noise_window(row["noise_profile"],int(row["seed"]),.05)
        parent=out["parent_raw"];bg=out["background"];reference=out["reference_sample"]
        kind="noise family source replay"
    raw_error=float(np.max(np.abs(parent-row["saved_parent"])))
    pre=parent[reference-100:reference];normalized=(parent[reference-100:reference+300]-float(pre.mean()))/(float(pre.std())+1e-8)
    norm_error=float(np.max(np.abs(normalized-row["saved_norm"])))
    ok=np.allclose(parent,row["saved_parent"],atol=reference_atol,rtol=reference_rtol) and np.allclose(normalized,row["saved_norm"],atol=reference_atol,rtol=reference_rtol)
    bgmatch=hashlib.sha256(bg.tobytes()).hexdigest()==row["background_sha256"]
    validation["replays"].append({"split":row["split"],"row":row["row"],"source_id":row["source_id"],
                                 "method":kind,"raw_max_difference":raw_error,"normalized_max_difference":norm_error,
                                 "background_hash_match":bgmatch,"pass":bool(ok and bgmatch)})
    assert ok and bgmatch,"CRITICAL independent replay mismatch"
# Revalidate actual saved normalization's response to future-only observed-sample edits.
future_diff=[]
for row in examples:
    p=int(row["reference_sample"]);a=row["saved_parent"]
    b=a.copy();b[p+300:]+=1e3
    pre=a[p-100:p];mu=float(pre.mean());sd=float(pre.std())+1e-8
    future_diff.append(float(np.max(np.abs(((a[p-100:p+300]-mu)/sd)-((b[p-100:p+300]-mu)/sd)))))
assert max(future_diff)==0.
validation["saved_preprocessing_future_max_difference"]=max(future_diff)
validation["independent_replay_tolerance"]={"absolute":reference_atol,"relative":reference_rtol,"reason":"Float32 independently ordered equations; pilot causality tolerance remains exactly0"}
validation["total_records"]=len(df_all)
validation["classification"]="Causal synthetic oracle-aligned research dataset, pending independent verification; not trigger-based streaming"
validation["training_allowed_by_this_agent"]=False
print("Independent stratified replays:",len(examples),"; maximum raw error",max(r["raw_max_difference"] for r in validation["replays"]),
      "; saved preprocessing prefix difference",max(future_diff))


Independent stratified replays: 29 ; maximum raw error 0.0 ; saved preprocessing prefix difference 0.0


In [4]:
# Final preservation and handoff. The implementation agent does not issue final approval.
after={rel:sha(ROOT/rel) for rel in before}
validation["protected_sources_unchanged"]={rel:after[rel]==before[rel] for rel in before}
assert all(validation["protected_sources_unchanged"].values())
savej(V3/"configs/protected_sources_after.json",after)
pilot=readj(V3/"reports/v3_counterfactual_results.json")
validation["pilot_status"]=pilot["status"]
validation["decision"]="READY FOR INDEPENDENT DATASET VERIFICATION"
eligibility={}
for t in [1,2,3]:
    eligible=sum(r["eligibility"][str(t)]["earthquake_eligible"] for r in validation["splits"].values())
    eligibility[str(t)]={"eligible":eligible,"earthquakes":8089,"fraction":eligible/8089}
validation["aggregate_eligibility"]=eligibility
savej(V3/"reports/v3_validation_results.json",validation)
def table(headers,rows):
    return "\n".join(["| "+" | ".join(headers)+" |","|"+"|".join(["---"]*len(headers))+"|"]+
                     ["| "+" | ".join(str(x) for x in row)+" |" for row in rows])
rounding_table=table(["Source","Unchanged S-P seconds","v2 rounded difference","v3 first-grid difference","2s/3s eligibility"],[
 (r["source_id"],f'{r["SP_seconds"]:.9f}',r["v2_nearest_SP_samples"],r["v3_first_grid_SP_samples"],f'{r["eligible2"]}/{r["eligible3"]}')
 for r in validation["rounding_cases"]])
report=f"""# Causal EEW v3 implementation validation

**READY FOR INDEPENDENT DATASET VERIFICATION.** This is implementation validation, not final dataset approval. No CNN/BiLSTM training, Optuna or fusion was performed.

## Root causes and repairs

V2's full-record background SD made early noise amplitude depend on future random innovations. Its finite-record P/S envelope maxima made earlier signal depend on future coda values. Exact v2 audit reproduction recovered normalized early-window differences **{pilot["v2_future_noise_max"]:.9f}** and **{pilot["v2_future_P_coda_max"]:.9f}**. Altering a future tail while preserving earlier innovations/envelopes is a valid intervention; altering shared magnitude/distance/decay parameters is not an invariance test.

V3 reuses the actual v2 earthquake implementation's velocities, Q(f), attenuation, source scaling, carriers and gamma time constants. Noise calibration now uses fixed impulse-response energy, independent of realized waveform samples. Gamma normalization uses the analytic peak at2*tau_d. Independent random streams separate onset, noise and background-profile parameters. Causal FIR pink-like noise replaces length-dependent FFT synthesis; transient disturbances have explicit causal onsets. These background changes are required causal repairs, not a claim of real-noise validation.

## Pilot gate

Fifteen stratified earthquake cases reproduce the exact v2 defects, then exercise S amplitude x4, future S coda +50%, future noise perturbation, future P coda +50% and x4,12/100/140-second durations, and postprocessing prefix tests. Sixteen noise cases cover four families and four seeds. **All v3 pilot prefix differences are exactly0 raw and normalized**, with a fixed zero tolerance. Full generation started only after this gate passed.

## Exhaustive saved-data checks

All **16,178 records**: train11,324, validation2,426, test2,428. HDF5 files save full100-second parents (N,10000), raw/normalized windows (N,200/300/400), pure-post windows (N,100/200/300),100Hz, float32 waveform arrays, separate metadata and explicit masks. Every numeric array is finite. Every window equals its parent slice and independently recomputed pre-context normalization; maximum normalization error={max(v["max_normalization_difference"] for v in validation["splits"].values())}.

All class/magnitude/source/station labels, original source IDs and split assignments, seeds, timestamps, P/S alignment and mask formulas pass. Parent waveform hashes and all saved window hashes show no cross-split duplicates; event, pair, parent and background identities are disjoint. Background digests are checked across every record and reproduced in stratified examples; full background arrays are not separately stored. {len(validation["replays"])} stratified replays pass, including a separate earthquake-equation implementation and each noise family. Maximum parent replay error={max(r["raw_max_difference"] for r in validation["replays"]):.9g}; maximum normalized replay error={max(r["normalized_max_difference"] for r in validation["replays"]):.9g}. Saved normalization's future-perturbation prefix difference=0.

Noise magnitude=-1 is explicitly excluded by magnitude_mask=(label==1); per-horizon magnitude masks also require pre-S eligibility. Poisoning noise targets leaves masked regression diagnostics unchanged. Noise has no fictional P/S prediction target: arrival labels remain-1. Noise pre-S flags are false; separate classification masks admit noise records. These are verified dataset contracts; a future training consumer must use the supplied masks.

## Arrival precision and pre-S eligibility

Continuous travel times are unchanged. S sample denotes the first grid sample at/after physical arrival (ceil), not a rounded-nearest index. Strict integer eligibility is reference_sample+100*tau < S_sample, with continuous-time agreement checked.

{rounding_table}

{table(["Horizon","Eligible earthquakes","Percentage"],[(t+"s",f'{v["eligible"]}/8089',f'{100*v["fraction"]:.4f}%') for t,v in eligibility.items()])}

No real arrival was moved to force eligibility. Raw parent P positions vary; oracle cropping deliberately puts P at index100 of positive windows. Negative references use the same3–7s range without a phase target. Future S is used only for offline eligibility labels; source/phase/magnitude metadata are not input channels.

## Limitations and preservation

This remains a **synthetic oracle-aligned benchmark**, not a streaming trigger/picker or operational EEW system. The1-second pre-P context is extra observed history beyond each1/2/3-second post-P horizon. Synthetic magnitudes affect early amplitudes under the source model, but single-station magnitude/distance tradeoffs, relative amplitude units, fixed propagation parameters, low-SNR examples and artificial background families limit identifiability and transfer to real data.

All **{len(before)} protected source/v2/audit files** have identical before/after SHA256, including original Phase11 data and CNN checkpoints. No existing status, memory, BiLSTM or original dataset was changed. New work is confined to eew_causal_v3.

Evidence: v3_counterfactual_results.json, v3_validation_results.json; ../configs/simulation_v3.json, split_manifest.csv, generation_manifest.json and protected_sources_before/after.json; ../dataset/eew_dataset_{{train,validation,test}}.h5; the three executed notebooks. No model training is authorized by this implementation result.
"""
(V3/"reports/v3_validation_report.md").write_text(report,encoding="utf-8")
handoff=f"""# Independent-review handoff: causal EEW v3

**Requested decision: independent dataset verification. Implementation status: READY FOR INDEPENDENT DATASET VERIFICATION; not finally approved.**

## Exact v2 evidence and v3 comparison

| Intervention | Reproduced v2 maximum normalized prefix difference | v3 maximum |
|---|---:|---:|
| Future noise only, after P+8s | {pilot["v2_future_noise_max"]:.9f} | 0 |
| Future P-coda only, after P+4s, x4 | {pilot["v2_future_P_coda_max"]:.9f} | 0 |
| Future S amplitude x4 before S | 0 | 0 |
| Future S coda +50%, future P coda +50%, record extension/truncation | Independently specified v3 gates | 0 |

The exact per-source v2 values are reproduced from the protected audit's instrument definition using the original saved-data generator. V3 tests preserve past source parameters and innovations; no tolerance was loosened (pilot tolerance0). Separate full-parent replay uses predetermined float32 tolerances atol1e-7/rtol1e-6, reported separately from the causality gate.

## Review targets

1. Inspect01_causal_simulator_v3.ipynb's narrow transformations of the v2 earthquake class: fixed filter gains, analytic gamma peaks, separate RNG domains and causal background profiles. Pilot code is in the same notebook.
2. Independently check02_generate_dataset_v3.ipynb's gate and immutable original assignments, source/magnitude/arrival metadata, seeds, full parents and explicit masks.
3. Re-run independent tests against saved HDF5 parents/windows.03_dataset_validation_v3.ipynb records exhaustive checks and a separate earthquake-equation replay, but was written by the implementation agent.
4. Check the two arrival-boundary records, source IDs NCS_EV_03038/NCS_EV_03572: v2 rounded differences200/300 become first-grid201/301 without changing continuous travel times. Verify strict integer masks; not all3s earthquake windows qualify.
5. Confirm magnitude loss uses magnitude_mask or magnitude_mask_{{horizon}} and never treats noise sentinel-1 as a magnitude target. Classification masks are separately provided.
6. Verify all before/after protected hashes and frozen generation hashes. Exact window and full-parent duplicate checks now cover saved parent arrays; background identities/digests are available, with full backgrounds reproducible by seeds rather than separately stored.

## Files and boundaries

- Executed notebooks: ../01_causal_simulator_v3.ipynb, ../02_generate_dataset_v3.ipynb, ../03_dataset_validation_v3.ipynb.
- Evidence: v3_counterfactual_results.json, v3_validation_results.json, v3_validation_report.md.
- Data: ../dataset/eew_dataset_{{train,validation,test}}.h5.
- Provenance: ../configs/split_manifest.csv; simulation_v3.json; generation_manifest.json; protected_sources_before.json; protected_sources_after.json; artifact_hashes.json.
- Protected v2 findings: ../../eew_causal_v2/codex_independent_audit/final_audit_report.md and audit_results.json.

Oracle P knowledge is still required to construct validation/test windows. No causal trigger has been implemented. Noise spectral realism and real-instrument magnitude calibration are not established. Review approval, if granted, can cover synthetic oracle-aligned research only—not operational EEW. No training or fusion was performed.
"""
(V3/"reports/v3_independent_review_handoff.md").write_text(handoff,encoding="utf-8")
savej(V3/"configs/artifact_hashes.json",{"files":{p.relative_to(V3).as_posix():sha(p) for p in V3.rglob("*") if p.is_file() and p.name!="artifact_hashes.json" and p.name!="03_dataset_validation_v3.ipynb"},
      "validation_notebook_excluded_from_self_hash":"execution output finalization; code source hash is recorded below",
      "validation_source_sha256":hashlib.sha256("\n".join("".join(c["source"]) for c in readj(V3/"03_dataset_validation_v3.ipynb")["cells"] if c["cell_type"]=="code").encode()).hexdigest()})
print(validation["decision"],"; eligibility",eligibility,"; protected hashes unchanged:",len(before))


READY FOR INDEPENDENT DATASET VERIFICATION ; eligibility {'1': {'eligible': 8089, 'earthquakes': 8089, 'fraction': 1.0}, '2': {'eligible': 8056, 'earthquakes': 8089, 'fraction': 0.9959203857089876}, '3': {'eligible': 7996, 'earthquakes': 8089, 'fraction': 0.9885029051798739}} ; protected hashes unchanged: 44
